# CHFJPY Z-Score Mean Reversion Strategy

**Logic:** Compute a 90-day rolling z-score of CHFJPY spot.
- Z < -2 → Long CHFJPY
- Z > +2 → Short CHFJPY
- Otherwise → Flat

**Backtest period:** 4 years  |  **Data source:** Bloomberg (`CHFJPY Curncy`)

In [ ]:
import pandas as pd
import numpy as np
from xbbg import blp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.edgecolor':   '#cccccc',
    'axes.labelcolor':  '#222222',
    'xtick.color':      '#444444',
    'ytick.color':      '#444444',
    'text.color':       '#222222',
    'grid.color':       '#e0e0e0',
    'grid.linestyle':   '-',
})

ACCENT = '#0066cc'

## 1. Data — Bloomberg

In [23]:
END   = datetime.today()
# Extra lookback so the 90-day window is fully warm at backtest start
START = END - timedelta(days=4*365 + 120)

raw = blp.bdh(
    tickers='CHFJPY Curncy',
    flds='PX_LAST',
    start_date=START.strftime('%Y-%m-%d'),
    end_date=END.strftime('%Y-%m-%d'),
)

# xbbg returns MultiIndex columns (ticker, field) — flatten to Series
spot = raw[('CHFJPY Curncy', 'PX_LAST')].dropna()
spot.index = pd.to_datetime(spot.index)
spot.name = 'CHFJPY'

print(f"Loaded {len(spot)} sessions  |  {spot.index[0].date()} → {spot.index[-1].date()}")
spot.tail(3)

Loaded 1128 sessions  |  2022-03-25 → 2026-07-22


2026-07-20    200.602
2026-07-21    200.780
2026-07-22    200.757
Name: CHFJPY, dtype: float64

## 2. Z-Score (90-day rolling)

In [24]:
WINDOW = 90

roll_mean = spot.rolling(WINDOW).mean()
roll_std  = spot.rolling(WINDOW).std()

zscore = (spot - roll_mean) / roll_std
zscore.name = 'zscore'

# Trim to the 4-year backtest window (fully-warm z-scores only)
backtest_start = END - timedelta(days=4*365)
spot_bt   = spot.loc[backtest_start:]
zscore_bt = zscore.loc[backtest_start:]

print(f"Backtest window: {spot_bt.index[0].date()} → {spot_bt.index[-1].date()}  ({len(spot_bt)} sessions)")

Backtest window: 2022-07-25 → 2026-07-22  (1042 sessions)


## 3. Signal & Position

In [25]:
THRESHOLD = 2.3

# Signal applied at close; position held the *next* day (no look-ahead)
signal = pd.Series(0, index=zscore_bt.index, dtype=float)
signal[zscore_bt < -THRESHOLD] =  1.0   # Long  (spot cheap vs 90d history)
signal[zscore_bt >  THRESHOLD] = -1.0   # Short (spot rich vs 90d history)

position = signal.shift(1).fillna(0)    # Enter next session's open (approx at prior close)

n_long  = (signal ==  1).sum()
n_short = (signal == -1).sum()
n_flat  = (signal ==  0).sum()
print(f"Long days: {n_long}  |  Short days: {n_short}  |  Flat days: {n_flat}")

Long days: 10  |  Short days: 43  |  Flat days: 989


## 4. PnL Calculation

In [26]:
# Daily log-return of CHFJPY spot
daily_ret = np.log(spot_bt / spot_bt.shift(1))

# Strategy daily PnL = position (lagged) * return
strat_ret = position * daily_ret

# Cumulative PnL in log-return space
cum_strat = strat_ret.cumsum()
cum_bh    = daily_ret.cumsum()     # Buy-and-hold benchmark

# ── Summary stats ──────────────────────────────────────────────
ann_factor = 252

total_ret = cum_strat.iloc[-1]
ann_ret   = strat_ret.mean() * ann_factor
ann_vol   = strat_ret.std()  * np.sqrt(ann_factor)
sharpe    = ann_ret / ann_vol if ann_vol != 0 else np.nan

running_max = cum_strat.cummax()
drawdown    = cum_strat - running_max
max_dd      = drawdown.min()

hit_rate = (strat_ret[position != 0] > 0).mean()

print("="*45)
print(f" Cumulative log-return : {total_ret:+.4f}  ({np.expm1(total_ret)*100:+.2f}%)")
print(f" Ann. return           : {ann_ret*100:+.2f}%")
print(f" Ann. volatility       : {ann_vol*100:.2f}%")
print(f" Sharpe ratio          : {sharpe:.2f}")
print(f" Max drawdown          : {max_dd*100:.2f}%")
print(f" Hit rate (in mkt)     : {hit_rate*100:.1f}%")
print("="*45)

 Cumulative log-return : +0.0310  (+3.14%)
 Ann. return           : +0.75%
 Ann. volatility       : 1.96%
 Sharpe ratio          : 0.38
 Max drawdown          : -2.87%
 Hit rate (in mkt)     : 50.9%


## 5. Charts

In [ ]:
fig = plt.figure(figsize=(14, 12), facecolor='white')
gs  = gridspec.GridSpec(4, 1, hspace=0.55)

# ── Panel 1: CHFJPY spot ────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.plot(spot_bt.index, spot_bt.values, color=ACCENT, lw=1.2)
ax1.set_title('CHFJPY Spot', fontsize=11)
ax1.set_ylabel('JPY per CHF')
ax1.grid(alpha=0.5)

# ── Panel 2: Z-Score + thresholds ───────────────────────────────
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.plot(zscore_bt.index, zscore_bt.values, color='#222222', lw=0.9, label='Z-score')
ax2.axhline( THRESHOLD, color='#cc0000', lw=1, ls='--', label=f'+{THRESHOLD}σ  (Short)')
ax2.axhline(-THRESHOLD, color='#009933', lw=1, ls='--', label=f'-{THRESHOLD}σ  (Long)')
ax2.axhline(0, color='#aaaaaa', lw=0.5)
ax2.fill_between(zscore_bt.index, zscore_bt, THRESHOLD,
                 where=zscore_bt > THRESHOLD,  alpha=0.15, color='#cc0000')
ax2.fill_between(zscore_bt.index, zscore_bt, -THRESHOLD,
                 where=zscore_bt < -THRESHOLD, alpha=0.15, color='#009933')
ax2.set_title('90-Day Rolling Z-Score', fontsize=11)
ax2.legend(fontsize=8, loc='upper right')
ax2.grid(alpha=0.5)

# ── Panel 3: Position ────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.fill_between(position.index, position.values, 0,
                 where=position > 0, color='#009933', alpha=0.7, label='Long')
ax3.fill_between(position.index, position.values, 0,
                 where=position < 0, color='#cc0000', alpha=0.7, label='Short')
ax3.set_yticks([-1, 0, 1])
ax3.set_yticklabels(['Short', 'Flat', 'Long'])
ax3.set_title('Position', fontsize=11)
ax3.legend(fontsize=8, loc='upper right')
ax3.grid(alpha=0.5)

# ── Panel 4: Cumulative PnL ──────────────────────────────────────
ax4 = fig.add_subplot(gs[3], sharex=ax1)
ax4.plot(cum_strat.index, cum_strat.values * 100, color=ACCENT, lw=1.4,
         label=f'Strategy  (SR={sharpe:.2f})')
ax4.plot(cum_bh.index,    cum_bh.values    * 100, color='#aaaaaa', lw=0.8,
         ls='--', label='Buy & Hold CHFJPY')
ax4.fill_between(drawdown.index, drawdown.values * 100, 0,
                 alpha=0.12, color='#cc0000')
ax4.axhline(0, color='#aaaaaa', lw=0.5)
ax4.set_title('Cumulative Log-Return (%)', fontsize=11)
ax4.set_ylabel('%')
ax4.legend(fontsize=8, loc='upper left')
ax4.grid(alpha=0.5)

fig.suptitle('CHFJPY Z-Score Mean-Reversion  |  ±2σ / 90d  |  4-Year Backtest  |  BBG',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('chfjpy_zscore_strategy.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Chart saved → chfjpy_zscore_strategy.png')

## 6. Annual PnL Breakdown

In [ ]:
annual = strat_ret.groupby(strat_ret.index.year).sum() * 100

fig, ax = plt.subplots(figsize=(8, 4), facecolor='white')
colors = ['#009933' if v >= 0 else '#cc0000' for v in annual.values]
ax.bar(annual.index.astype(str), annual.values, color=colors, edgecolor='#333333', lw=0.4)
ax.axhline(0, color='#333333', lw=0.6)
for i, (yr, val) in enumerate(annual.items()):
    ax.text(i, val + (0.3 if val >= 0 else -0.6), f'{val:+.2f}%',
            ha='center', va='bottom' if val >= 0 else 'top', fontsize=9, color='#222222')
ax.set_title('Annual PnL — CHFJPY Z-Score Strategy', fontsize=11)
ax.set_ylabel('Log-Return (%)')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('chfjpy_annual_pnl.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(annual.to_string())